# Argo CLI Usage Tutorial

This notebook demonstrates how to use the **Argo CLI** (`argo` command) for Monte Carlo simulations via configuration files.

The Argo CLI provides four main commands:
1. **`argo distributions`** - List all available probability distributions
2. **`argo generate`** - Create template configuration files
3. **`argo validate`** - Validate configuration files against schema
4. **`argo simulate`** - Run Monte Carlo simulations from config files

## Installation

The CLI is available in the `@argo/cli` package:

```bash
npm install -g @argo/cli
```

Or run locally from the monorepo:

```bash
node packages/argo-cli/bin/argo.js <command>
```

## Part 1: List Available Distributions

The `argo distributions` command lists all 14 probability distributions with their parameters.

In [ ]:
const { execSync } = require('child_process');
const path = require('path');

// Path to CLI (relative to notebooks directory)
const cliPath = path.join('..', 'packages', 'argo-cli', 'bin', 'argo.js');

// Execute argo distributions command
const output = execSync(`node ${cliPath} distributions`, { encoding: 'utf-8' });
console.log(output);

## Part 2: Generate Configuration Template

The `argo generate` command creates a template configuration file with example simulation setup.

In [ ]:
const fs = require('fs');

// Generate template config
const configPath = 'example-simulation.json';
try {
  fs.unlinkSync(configPath); // Remove if exists
} catch (err) {}

const generateOutput = execSync(`node ${cliPath} generate ${configPath}`, { encoding: 'utf-8' });
console.log(generateOutput);

// Display generated config
const config = JSON.parse(fs.readFileSync(configPath, 'utf-8'));
console.log('\nGenerated configuration:');
console.log(JSON.stringify(config, null, 2));

## Part 3: Validate Configuration

The `argo validate` command checks configuration files against the JSON schema to ensure they are valid.

In [ ]:
// Validate the generated config
const validateOutput = execSync(`node ${cliPath} validate ${configPath}`, { encoding: 'utf-8' });
console.log(validateOutput);

### Example: Invalid Configuration

Let's see what happens when we validate an invalid config:

In [ ]:
// Create invalid config (missing required field)
const invalidConfig = {
  // Missing iterations field
  variables: [
    {
      name: 'X',
      type: 'input',
      distribution: {
        type: 'Normal',
        parameters: { mean: 100, stddev: 15 }
      }
    }
  ]
};

const invalidPath = 'invalid-config.json';
fs.writeFileSync(invalidPath, JSON.stringify(invalidConfig, null, 2));

try {
  execSync(`node ${cliPath} validate ${invalidPath}`, { encoding: 'utf-8' });
} catch (err) {
  console.log('Validation failed (as expected):');
  console.log(err.stderr);
}

// Clean up
fs.unlinkSync(invalidPath);

## Part 4: Run Simple Simulation

The `argo simulate` command runs Monte Carlo simulations from configuration files.

In [ ]:
// Run simulation from generated config
const simulateOutput = execSync(`node ${cliPath} simulate ${configPath}`, { encoding: 'utf-8' });
console.log(simulateOutput);

## Part 5: Custom Configuration - Project Risk Analysis

Let's create a custom configuration for a software project risk analysis with correlated tasks.

In [ ]:
// Create custom project risk config
const projectConfig = {
  iterations: 10000,
  seed: 42,
  variables: [
    {
      name: 'Frontend',
      type: 'input',
      distribution: {
        type: 'PERT',
        parameters: { min: 8, mode: 12, max: 20 }
      }
    },
    {
      name: 'Backend',
      type: 'input',
      distribution: {
        type: 'PERT',
        parameters: { min: 10, mode: 15, max: 25 }
      }
    },
    {
      name: 'Testing',
      type: 'input',
      distribution: {
        type: 'Triangular',
        parameters: { min: 4, mode: 6, max: 10 }
      }
    },
    {
      name: 'WeeklyCost',
      type: 'input',
      distribution: {
        type: 'Normal',
        parameters: { mean: 10000, stddev: 1000 }
      }
    },
    {
      name: 'Duration',
      type: 'formula',
      formula: 'Frontend + Backend + Testing'
    },
    {
      name: 'TotalCost',
      type: 'formula',
      formula: 'Duration * WeeklyCost'
    }
  ],
  correlations: {
    'Frontend-Backend': 0.6,
    'Backend-Testing': 0.5
  },
  outputs: ['Duration', 'TotalCost']
};

const projectPath = 'project-risk.json';
fs.writeFileSync(projectPath, JSON.stringify(projectConfig, null, 2));

console.log('Created project risk configuration:');
console.log(JSON.stringify(projectConfig, null, 2));

### Validate and Run Project Simulation

In [ ]:
// Validate project config
console.log('=== VALIDATION ===');
const projectValidate = execSync(`node ${cliPath} validate ${projectPath}`, { encoding: 'utf-8' });
console.log(projectValidate);

// Run project simulation
console.log('\n=== SIMULATION ===');
const projectSimulate = execSync(`node ${cliPath} simulate ${projectPath}`, { encoding: 'utf-8' });
console.log(projectSimulate);

## Part 6: JSON Output Mode

The CLI can output results in JSON format for programmatic processing using the `--output` flag.

In [ ]:
const jsonOutputPath = 'project-results.json';

// Run simulation with JSON output
execSync(`node ${cliPath} simulate ${projectPath} --output ${jsonOutputPath}`);

// Load and display results
const output = JSON.parse(fs.readFileSync(jsonOutputPath, 'utf-8'));
const results = output.results.statistics;

console.log('Project Duration Statistics:');
console.log(`  Mean: ${results.Duration.mean.toFixed(2)} weeks`);
console.log(`  Median: ${results.Duration.median.toFixed(2)} weeks`);
console.log(`  Std Dev: ${results.Duration.stdDev.toFixed(2)} weeks`);
console.log(`  90% CI: [${results.Duration.p5.toFixed(2)}, ${results.Duration.p95.toFixed(2)}] weeks`);

console.log('\nProject Cost Statistics:');
console.log(`  Mean: $${results.TotalCost.mean.toFixed(0)}`);
console.log(`  Median: $${results.TotalCost.median.toFixed(0)}`);
console.log(`  Std Dev: $${results.TotalCost.stdDev.toFixed(0)}`);
console.log(`  90% CI: [$${results.TotalCost.p5.toFixed(0)}, $${results.TotalCost.p95.toFixed(0)}]`);

## Part 7: Configuration Schema Reference

Here's the complete schema for Argo configuration files:

In [ ]:
const schemaExample = {
  iterations: 10000,          // Required: Number of Monte Carlo iterations
  seed: 42,                   // Optional: Random seed for reproducibility
  
  variables: [                // Required: Array of simulation variables
    {
      name: 'VariableName',   // Required: Unique variable name
      type: 'input',          // Required: 'input' or 'formula'
      
      // For input variables:
      distribution: {
        type: 'Normal',       // Required: Distribution type (capitalized)
        parameters: {         // Required: Distribution-specific parameters
          mean: 100,
          stddev: 15
        }
      }
    },
    {
      name: 'Output',
      type: 'formula',        // Formula variable
      formula: 'VariableName * 2'  // Required for formula type
    }
  ],
  
  correlations: {             // Optional: Variable correlations
    'Var1-Var2': 0.7          // Key format: 'var1-var2', value: correlation [-1, 1]
  },
  
  outputs: ['Output']         // Optional: Variables to report (defaults to all)
};

console.log('Configuration Schema:');
console.log(JSON.stringify(schemaExample, null, 2));

### Supported Distribution Types

All 14 distributions from `@argo/core` are supported:

In [ ]:
const distributionExamples = [
  // Continuous distributions
  { type: 'Normal', parameters: { mean: 100, stddev: 15 } },
  { type: 'LogNormal', parameters: { mu: 4, sigma: 0.5 } },
  { type: 'Uniform', parameters: { min: 0, max: 100 } },
  { type: 'Triangular', parameters: { min: 50, mode: 100, max: 200 } },
  { type: 'PERT', parameters: { min: 50, mode: 100, max: 200 } },
  { type: 'Exponential', parameters: { lambda: 0.5 } },
  { type: 'Gamma', parameters: { shape: 2, scale: 2 } },
  { type: 'Beta', parameters: { alpha: 2, beta: 5 } },
  { type: 'Weibull', parameters: { shape: 2, scale: 100 } },
  { type: 'Pareto', parameters: { shape: 2, scale: 1 } },
  
  // Discrete distributions
  { type: 'Binomial', parameters: { n: 20, p: 0.3 } },
  { type: 'Poisson', parameters: { lambda: 5 } },
  { type: 'Geometric', parameters: { p: 0.3 } },
  { type: 'Hypergeometric', parameters: { N: 50, K: 20, n: 10 } }
];

console.log('Distribution Configuration Examples:');
distributionExamples.forEach(dist => {
  console.log(`\n${dist.type}:`);
  console.log(`  ${JSON.stringify(dist)}`);
});

## Part 8: Clean Up Test Files

In [ ]:
// Clean up generated files
const filesToRemove = [configPath, projectPath, jsonOutputPath];

filesToRemove.forEach(file => {
  try {
    fs.unlinkSync(file);
    console.log(`Removed: ${file}`);
  } catch (err) {
    // File might not exist, ignore
  }
});

console.log('\nCleanup complete!');

## Summary

The Argo CLI provides a complete workflow for Monte Carlo simulations:

1. **`argo distributions`** - Discover available probability distributions
2. **`argo generate`** - Create template configuration files
3. **`argo validate`** - Validate configurations against JSON schema
4. **`argo simulate`** - Run simulations with formatted or JSON output

**Key Features:**
- All 14 probability distributions supported
- Formula variables with automatic dependency resolution
- Correlation modeling with Gaussian copula
- JSON schema validation for configuration files
- Both human-readable and JSON output formats
- Reproducible results with seeded RNG

**Next Steps:**
- Explore the other tutorial notebooks (01-distributions, 02-statistics, 03-risk-analysis, 04-monte-carlo)
- Create your own configuration files for real-world simulations
- Integrate Argo CLI into your project risk management workflows